# Morfologia matemática binária

Este notebook apresenta operações morfológicas em máscaras binárias: erosão, dilatação, abertura, fechamento e gradiente.

**Público e pré-requisitos:** estudantes que já conhecem imagens NumPy, vizinhanças e operações locais.

## Objetivos e roteiro

Ao final, você conseguirá criar elementos estruturantes, prever o efeito de cada operação e escolher uma operação para remover ruído, preencher lacunas ou destacar fronteiras.

1. Definir a convenção binária.
2. Comparar elementos estruturantes.
3. Aplicar erosão e dilatação.
4. Aplicar abertura, fechamento e gradiente.
5. Experimentar uma máscara própria.

## 1. Instalação

A instalação usa a versão commitada na branch principal do repositório.

In [ ]:
%pip install -q "git+https://github.com/tfvieira/dip-2026-2.git"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from dip_toolkit.modules.morphology import MorphologyProcessor

morphology = MorphologyProcessor()

## 2. Convenção binária

As operações aceitam arrays 2D `uint8`: `0` representa o fundo e `255` representa o primeiro plano. Cada resultado preserva shape, dtype e esses dois valores.

Na borda `constant`, pixels fora da imagem são fundo; em `replicate`, repete-se o pixel mais próximo.

In [ ]:
mask = np.zeros((11, 11), dtype=np.uint8)
mask[3:8, 3:8] = 255
mask[1, 1] = 255  # pequeno ruído
mask[5, 5] = 0  # lacuna

print(f"shape={mask.shape}, dtype={mask.dtype}, valores={np.unique(mask)}")
plt.figure(figsize=(4, 4))
plt.imshow(mask, cmap="gray", vmin=0, vmax=255)
plt.title("Máscara binária inicial")
plt.axis("off")
plt.show()

## 3. Elementos estruturantes

O elemento estruturante define a vizinhança observada. Dimensões são ímpares para que exista um centro. O retângulo considera toda a janela, a cruz privilegia direções horizontal e vertical, e a elipse aproxima uma vizinhança circular.

In [ ]:
elements = {
    "Retângulo": morphology.create_structuring_element("rectangle", (5, 5)),
    "Elipse": morphology.create_structuring_element("ellipse", (5, 5)),
    "Cruz": morphology.create_structuring_element("cross", (5, 5)),
}
figure, axes = plt.subplots(1, 3, figsize=(10, 3))
for axis, (name, element) in zip(axes, elements.items(), strict=True):
    axis.imshow(element, cmap="gray", vmin=0, vmax=1)
    axis.set_title(name)
    axis.axis("off")
figure.tight_layout()
plt.show()

## 4. Erosão e dilatação

A erosão mantém primeiro plano apenas quando o elemento cabe inteiramente nele, reduzindo objetos. A dilatação amplia o primeiro plano quando alguma posição ativa do elemento o encontra.

In [ ]:
element = elements["Cruz"]
eroded = morphology.erode(mask, element, border="constant")
dilated = morphology.dilate(mask, element, border="constant")

figure, axes = plt.subplots(1, 3, figsize=(11, 4))
for axis, image, title in zip(
    axes,
    (mask, eroded, dilated),
    ("Original", "Erosão", "Dilatação"),
    strict=True,
):
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()
plt.show()

## 5. Abertura, fechamento e gradiente

A abertura é erosão seguida de dilatação e tende a remover ruídos pequenos. O fechamento é dilatação seguida de erosão e tende a preencher lacunas. O gradiente é `dilatação - erosão`, destacando a região de fronteira.

In [ ]:
opened = morphology.opening(mask, element)
closed = morphology.closing(mask, element)
gradient = morphology.gradient(mask, element)

figure, axes = plt.subplots(1, 3, figsize=(11, 4))
for axis, image, title in zip(
    axes,
    (opened, closed, gradient),
    ("Abertura", "Fechamento", "Gradiente"),
    strict=True,
):
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()
plt.show()

## Exercício e atenção

Crie uma máscara com dois objetos, um ruído e uma lacuna. Compare o resultado da abertura e do fechamento usando elementos `rectangle` e `cross` de tamanhos 3 e 5.

**Erro comum:** usar valores `0` e `1` como máscara. A API exige `uint8` com `0` e `255`; converta, por exemplo, com `(condition.astype(np.uint8) * 255)`.

Como extensão, investigue top-hat e black-hat após compreender a composição das operações básicas.

In [ ]:
# TODO: altere a máscara e compare os dois elementos estruturantes.
student_mask = mask.copy()
student_element = morphology.create_structuring_element("rectangle", (3, 3))
student_result = morphology.opening(student_mask, student_element)

np.array_equal(student_result, opened)